# ch05 运行效能评估

**目标**：计算正常与故障场景下的分拣效率，分析数据集局限性

**依赖**：ch03, ch04

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('.') / '..'))
from src.utils.data_loader import load_anomaly_data, load_state_machine_data
from src.utils.output_manager import save_dataframe, save_markdown

import pandas as pd
import numpy as np

CHAPTER_ID = 'ch05'

## 1. 数据加载

In [ ]:
anomaly_df = load_anomaly_data()
state_df = load_state_machine_data()
state_df['Label'] = state_df['Label'].astype(int)
print(f"异常数据: {len(anomaly_df)} 行")
print(f"状态数据: {len(state_df)} 行")

## 2. 分拣周期识别

In [ ]:
state_changes = state_df['Label'].diff().ne(0)
change_points = state_df.index[state_changes]

cycle_durations = []
for i in range(len(change_points) - 1):
    duration = (change_points[i + 1] - change_points[i]).total_seconds()
    cycle_durations.append({'cycle_duration': duration})

cycle_df = pd.DataFrame(cycle_durations)
print(f"共识别 {len(cycle_df)} 个分拣周期")
cycle_df.describe()

## 3. 效率计算

In [ ]:
total_samples = len(anomaly_df)
anomaly_count = len(anomaly_df[anomaly_df['Label'] != 0])

efficiency_stats = {
    'total_samples': total_samples,
    'anomaly_count': anomaly_count,
    'anomaly_rate': anomaly_count / total_samples,
    'normal_rate': (total_samples - anomaly_count) / total_samples,
}

pd.DataFrame([efficiency_stats])

## 4. 稳定性统计

In [ ]:
normal_mask = anomaly_df['Label'] == 0
normal_runs = []
current_run = 0

for is_normal in normal_mask:
    if is_normal:
        current_run += 1
    else:
        if current_run > 0:
            normal_runs.append(current_run)
            current_run = 0
if current_run > 0:
    normal_runs.append(current_run)

stability_stats = {
    'max_consecutive_normal': max(normal_runs) if normal_runs else 0,
    'mean_consecutive_normal': np.mean(normal_runs) if normal_runs else 0,
}

pd.DataFrame([stability_stats])

## 5. 数据集局限性分析

In [ ]:
limitations = [
    "1. 异常样本极少（仅 50 条，占 0.3%），统计代表性有限",
    "2. 仅包含两种故障类型（卡滞、脱扣），未覆盖全部故障场景",
    "3. 数据来自单一时间段，缺乏季节性/长期趋势信息",
    "4. 信号类型有限（仅电机和 PLC I/O），缺少视觉/温度等传感器",
    "5. 物料种类单一，未覆盖不同材质/形状的零件",
]

for lim in limitations:
    print(lim)

## 6. 保存产物

In [ ]:
save_dataframe(cycle_df, 'sorting_cycle_stats.csv', CHAPTER_ID)
save_dataframe(pd.DataFrame([efficiency_stats]), 'efficiency_comparison_table.csv', CHAPTER_ID)
save_dataframe(pd.DataFrame([stability_stats]), 'system_stability_assessment.csv', CHAPTER_ID)

limitations_content = "\n".join(limitations)
save_markdown(limitations_content, 'dataset_limitations.md', CHAPTER_ID)